# Candidate generation: rolling popularity + content two-tower (750 / 750)

Combined candidate generator with a fixed per-request budget of **1,500 articles**:

- **Slots 1-750:** rolling prior-hour click popularity, exactly the
  `prior_hour_clicks` order from `CG_rolling_popularity_baseline.ipynb`.
- **Slots 751-1500:** content two-tower from `CG_content_two_tower_mlp.ipynb`,
  scoring the **full eligible catalogue** rather than a popularity Top-K pool.
  The content model uses only intrinsic article content; there are no
  popularity inputs in either tower. Rolling prior-hour clicks are used only for
  content-score tie-breaking and empty-profile fallback.

Protocol:

- A request in hour `h` uses only completed hours `< h` for popularity.
- Both halves exclude every article in the request's `full_history_rows`.
- Eligibility is `available_from <= timestamp`.
- The content half is a personalised ranking over the full time-eligible
  catalogue. This deliberately trades some recall against popularity for better
  downstream personalisation and niche-article coverage.
- The popularity half and content half are disjoint, so every impression gets
  exactly 1,500 unique candidates.
- The content model is trained with BPR on the full `labels_train` pool, using each
  clicked positive and up to four exposed negatives redrawn per epoch from the same request.
- Evaluation uses the shared candidate evaluator on the combined candidate set.

The notebook trains on all training groups and scores the complete target split.


## 1. Setup


In [1]:
from __future__ import annotations

import json
import sys
import time
from pathlib import Path

import numpy as np
import polars as pl
import torch
import torch.nn.functional as F
from torch import nn

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run_all.py").is_file()
)
CG_DIR = ROOT / "5_Models" / "5_01_Candidate_Generation"
sys.path.insert(0, str(CG_DIR))
sys.path.insert(0, str(ROOT / "src"))

from cf_models import resolve_device
from cg_result_writer import CandidateWriter
from neural_helpers import (
    EarlyStopper,
    EpochNegativeSampler,
    chronological_cutoff,
    chronological_mask,
    max_clicks_per_impression,
    mean_slate_metrics,
    tuning_slates,
)
from newsrec.common import Settings

PROCESSED = ROOT / "1_Data_Preparation" / "processed"
FEATURES = ROOT / "4_Feature_Engineering" / "artifacts" / "prequential-all"
REPRESENTATIONS = ROOT / "3_Representations" / "artifacts"
ARTIFACTS = ROOT / "5_Models" / "5_01_Candidate_Generation" / "artifacts"
RESULTS = ROOT / "results"

TARGET_SPLIT = "validation"
FIT_SPLIT = "train"
SEED = 4222

TOTAL_BUDGET = 500
POPULARITY_BUDGET = 200
CONTENT_BUDGET = 300
CANDIDATE_CAP = TOTAL_BUDGET
M_VALUES = (500,)

POPULARITY_SCAN = 4_000

HIDDEN_DIM = 256
EMBED_DIM = 128
DROPOUT = 0.1
NUM_EPOCHS = 50
BATCH_SIZE = 16_384
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
PATIENCE = 7
TUNE_QUANTILE = 0.9
TUNE_CANDIDATES = PROCESSED / "candidates_train.parquet"
DEVICE = resolve_device("auto")

HOUR_NS = 3_600 * 10**9
POPULARITY_SPLITS = (
    ["train", "validation"]
    if TARGET_SPLIT == "validation"
    else ["train", "validation", "test"]
)

MODEL_NAME = "content_two_tower_rolling_popularity_750_750"
RUN_TAG = "combined_content_two_tower_rolling_popularity"
ARTIFACT_DIR = ARTIFACTS / RUN_TAG
RECOMMENDATIONS_PATH = (
    ARTIFACT_DIR / f"recommendations_{TARGET_SPLIT}_top{CANDIDATE_CAP}.parquet"
)
RESULT_PATH = RESULTS / f"candidate_generation_{MODEL_NAME}_{TARGET_SPLIT}.csv"

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print("Repository:", ROOT)
print("Target split:", TARGET_SPLIT)
print(
    "Budget:",
    POPULARITY_BUDGET,
    "popularity +",
    CONTENT_BUDGET,
    "content =",
    TOTAL_BUDGET,
)
print("Device:", DEVICE)
print("Torch:", torch.__version__)


Repository: c:\Users\Yong Rui\BT4222-Group-6-
Target split: validation
Budget: 200 popularity + 300 content = 500
Device: cuda
Torch: 2.13.0a0+rocm7.13.0a20260416


### Shared dimensions


In [2]:
TEXT_DIM = 64
ENTITY_DIM = 100
CONTENT_DIM = TEXT_DIM + ENTITY_DIM + 2
INPUT_DIM = CONTENT_DIM
ITEM_INPUT_DIM = CONTENT_DIM


## 2. Data ingestion and model-related processing


### History pooling and feature builders

`pool_profiles` mean-pools valid article vectors and L2-normalises each block.
`profile_features` builds the 166-dimensional request input. `content_features`
builds the 166-dimensional intrinsic article-content block, which is the only
input to the item tower.


In [3]:
def pool_profiles(history_rows, matrix, block=2_000):
    matrix = np.asarray(matrix)
    n = len(history_rows)
    dim = matrix.shape[1]
    result = np.zeros((n, dim), dtype=np.float32)
    for start in range(0, n, block):
        chunk = history_rows[start : start + block]
        lengths = np.fromiter(
            (len(rows) for rows in chunk), dtype=np.int64, count=len(chunk)
        )
        total = int(lengths.sum())
        if total == 0:
            continue
        flat = np.concatenate(
            [np.asarray(rows, dtype=np.int64) for rows in chunk if len(rows)]
        )
        vectors = np.asarray(matrix[flat], dtype=np.float32)
        valid = np.linalg.norm(vectors, axis=1) > 1e-12
        if not valid.any():
            continue

        groups = np.repeat(np.arange(len(chunk), dtype=np.int64), lengths)
        valid_groups = groups[valid]
        valid_vectors = vectors[valid]
        starts = np.r_[
            0, np.flatnonzero(valid_groups[1:] != valid_groups[:-1]) + 1
        ]
        sums = np.add.reduceat(valid_vectors, starts, axis=0)
        counts = np.diff(np.r_[starts, len(valid_groups)])
        pooled = sums / counts[:, None]
        norms = np.linalg.norm(pooled, axis=1)
        nonzero = norms > 0
        pooled[nonzero] /= norms[nonzero, None]
        result[start : start + len(chunk)][valid_groups[starts]] = pooled
    return result


In [4]:
def profile_features(history_rows, text, entity, block=2_000):
    n = len(history_rows)
    out = np.zeros((n, INPUT_DIM), dtype=np.float32)
    for start in range(0, n, block):
        chunk = history_rows[start : start + block]
        text_profile = pool_profiles(chunk, text, block=len(chunk))
        entity_profile = pool_profiles(chunk, entity, block=len(chunk))
        stop = start + len(chunk)
        out[start:stop, :TEXT_DIM] = text_profile
        out[start:stop, TEXT_DIM : TEXT_DIM + ENTITY_DIM] = entity_profile
        out[start:stop, -2] = np.linalg.norm(text_profile, axis=1) > 0
        out[start:stop, -1] = np.linalg.norm(entity_profile, axis=1) > 0
    return out


In [5]:
def content_features(text, entity):
    text = np.asarray(text, dtype=np.float32)
    entity = np.asarray(entity, dtype=np.float32)
    n = text.shape[0]
    out = np.zeros((n, CONTENT_DIM), dtype=np.float32)
    out[:, :TEXT_DIM] = text
    out[:, TEXT_DIM : TEXT_DIM + ENTITY_DIM] = entity
    out[:, CONTENT_DIM - 2] = np.linalg.norm(text, axis=1) > 0
    out[:, CONTENT_DIM - 1] = np.linalg.norm(entity, axis=1) > 0
    return out


def item_inputs(rows):
    return np.asarray(CONTENT_INPUTS[rows], dtype=np.float32)


### Causal request-profile caches

The profile row for each impression contains the articles clicked before that
request. The cache stores one 166-dimensional input per request: `text64`
profile, `entity100` profile and two availability flags. Repeated history
references are preserved, zero vectors are ignored and each block is
L2-normalised. The profile index also supplies the
`impression_uid -> profile_row` map used to join training rows.


In [6]:
def load_profiles(split):
    parts = sorted((FEATURES / f"request_profiles_{split}").glob("*.parquet"))
    return pl.read_parquet(
        parts,
        columns=["impression_uid", "timestamp", "full_history_rows"],
    ).sort("timestamp", maintain_order=True)


In [7]:
def profile_cache_paths(split):
    return (
        ARTIFACT_DIR / f"user_features_{split}.npy",
        ARTIFACT_DIR / f"profile_uids_{split}.npy",
        ARTIFACT_DIR / f"profile_index_{split}.parquet",
    )


In [8]:
def load_profile_cache(split):
    feature_path, uid_path, index_path = profile_cache_paths(split)
    if feature_path.exists() and uid_path.exists() and index_path.exists():
        return np.load(feature_path, mmap_mode="r"), np.load(uid_path)

    frame = load_profiles(split)
    n = frame.height
    features = np.lib.format.open_memmap(
        feature_path, mode="w+", dtype=np.float32, shape=(n, INPUT_DIM)
    )
    for start in range(0, n, 20_000):
        end = min(start + 20_000, n)
        rows = frame["full_history_rows"].slice(start, end - start).to_list()
        features[start:end] = profile_features(rows, TEXT, ENTITY)
    features.flush()
    uids = frame["impression_uid"].to_numpy()
    np.save(uid_path, uids)
    pl.DataFrame(
        {
            "impression_uid": uids,
            "profile_row": np.arange(n, dtype=np.int32),
        }
    ).write_parquet(index_path)
    print(f"Built profile cache for {split}: {n:,} requests")
    return np.load(feature_path, mmap_mode="r"), uids


In [9]:
TEXT = np.load(REPRESENTATIONS / "text64.npy", mmap_mode="r")

ENTITY = np.load(REPRESENTATIONS / "article_entity100.npy", mmap_mode="r")

TRAIN_USER_FEATURES, _ = load_profile_cache(FIT_SPLIT)

PROFILE_INDEX_TRAIN = pl.read_parquet(profile_cache_paths(FIT_SPLIT)[2])

print(f"Train requests: {TRAIN_USER_FEATURES.shape[0]:,}")

print(f"Train user feature shape: {TRAIN_USER_FEATURES.shape}")


Built profile cache for train: 95,071 requests
Train requests: 95,071
Train user feature shape: (95071, 166)


### Article content inputs and the time-eligible catalogue

Article vectors are indexed by `article_row`, with row 0 reserved as padding.
The item tower receives only intrinsic content vectors and missing-vector flags.
Rolling popularity is not an item-tower feature; it is used only for candidate
tie-breaking and the empty-history fallback. The catalogue is ordered by
`available_from` so each request timestamp selects a prefix.


In [10]:
article_index = pl.read_parquet(
    REPRESENTATIONS / "article_index.parquet", columns=["article_row", "news_code"]
).sort("article_row")
n_items = article_index.height + 1

availability = pl.read_parquet(
    REPRESENTATIONS / "article_availability.parquet",
    columns=["news_code", "available_from"],
)
catalogue = article_index.join(availability, on="news_code", how="left")
ordered = catalogue.sort(["available_from", "news_code"])
ARTICLE_ROWS_SORTED = ordered["article_row"].to_numpy().astype(np.int64)
NEWS_SORTED = ordered["news_code"].to_numpy().astype(np.int64)
AVAILABLE_SORTED = ordered["available_from"].to_numpy().astype("datetime64[ns]")

CONTENT_INPUTS = content_features(TEXT, ENTITY)
CONTENT_INPUTS_SORTED = CONTENT_INPUTS[ARTICLE_ROWS_SORTED]
POS_IN_SORTED = np.full(n_items, -1, dtype=np.int64)
POS_IN_SORTED[ARTICLE_ROWS_SORTED] = np.arange(
    len(ARTICLE_ROWS_SORTED), dtype=np.int64
)

print(f"Catalogue articles: {article_index.height:,}")
print(f"Content input shape: {CONTENT_INPUTS.shape}")


Catalogue articles: 51,282
Content input shape: (51283, 166)


## 3. Rolling prior-hour popularity state

Hourly exposures and clicks are aggregated from the logged candidate slates. A request in hour `h` uses only completed hours `< h`. The popularity half matches `CG_rolling_popularity_baseline.ipynb`: prior-hour clicks descending, prior-hour exposures descending, then `news_code` ascending. The content half uses the clicked-only prior-hour order from `CG_content_two_tower_mlp.ipynb` for tie-breaking and empty-profile fallback. Popularity is never an input to either tower.


In [11]:
first_ts = (
    pl.scan_parquet(PROCESSED / "candidates_train.parquet")
    .select(pl.col("timestamp").min())
    .collect()
    .item()
)
T0_NS = int(
    np.datetime64(first_ts, "ns")
    .astype("datetime64[D]")
    .astype("datetime64[ns]")
    .astype(np.int64)
)

hourly = (
    pl.concat(
        [
            pl.scan_parquet(PROCESSED / f"candidates_{split}.parquet")
            .select(
                "news_code",
                "clicked",
                (
                    (
                        (pl.col("timestamp").dt.epoch("ns") - T0_NS)
                        // HOUR_NS
                    ).cast(pl.Int64)
                ).alias("hour"),
            )
            for split in POPULARITY_SPLITS
        ]
    )
    .group_by("news_code", "hour")
    .agg(
        exp=pl.len(),
        clk=pl.col("clicked").cast(pl.Int64).sum(),
    )
    .collect()
)

index_codes = article_index["news_code"].to_numpy().astype(np.int64)
index_rows = article_index["article_row"].to_numpy().astype(np.int64)
row_of_code = np.zeros(int(index_codes.max()) + 1, dtype=np.int64)
row_of_code[index_codes] = index_rows
row_to_code = np.zeros(n_items, dtype=np.int32)
row_to_code[index_rows] = index_codes.astype(np.int32)

source_codes = hourly["news_code"].to_numpy().astype(np.int64)
source_rows = row_of_code[source_codes]

N_HOURS = int(hourly["hour"].max()) + 1
EXPOSURES = np.zeros((N_HOURS, n_items), dtype=np.int32)
CLICKS = np.zeros((N_HOURS, n_items), dtype=np.int32)
EXPOSURES[hourly["hour"].to_numpy(), source_rows] = hourly["exp"].to_numpy()
CLICKS[hourly["hour"].to_numpy(), source_rows] = hourly["clk"].to_numpy()

source_total = int(EXPOSURES.sum())

avail_row = np.full(n_items, np.iinfo(np.int64).max, dtype=np.int64)
avail_row[ARTICLE_ROWS_SORTED] = (
    AVAILABLE_SORTED.astype("datetime64[ns]").astype(np.int64)
)

MIN_HOUR = int(T0_NS // HOUR_NS)


def hour_index_of(timestamps_ns):
    return np.asarray(timestamps_ns, dtype=np.int64) // HOUR_NS - MIN_HOUR


def hourly_popularity(hour, item_rows):
    if hour <= 0:
        return np.zeros(len(item_rows), dtype=np.float32)
    return np.log1p(CLICKS[hour - 1, item_rows]).astype(np.float32)


def score_snapshot(hour):
    hour = int(hour)
    zero = np.zeros(n_items, dtype=np.float32)
    prior_exp = (
        np.log1p(EXPOSURES[hour - 1]).astype(np.float32)
        if hour > 0
        else zero
    )
    prior_clk = (
        np.log1p(CLICKS[hour - 1]).astype(np.float32)
        if hour > 0
        else zero
    )
    return prior_clk, prior_exp


def popularity_snapshot(hour):
    hour = int(hour)
    hour_end = T0_NS + (hour + 1) * HOUR_NS
    eligible = np.flatnonzero(avail_row <= hour_end)
    eligible = eligible[eligible != 0]
    score, tiebreak = score_snapshot(hour)
    order = eligible[
        np.lexsort(
            (row_to_code[eligible], -tiebreak[eligible], -score[eligible])
        )
    ]
    return order, POS_IN_SORTED[order]


print(f"Source candidate rows: {source_total:,}")
print(f"Rolling popularity hours: {N_HOURS:,}")
print(f"Articles clicked at least once: {int((CLICKS.sum(0) > 0).sum()):,}")


Source candidate rows: 4,621,015
Rolling popularity hours: 120
Articles clicked at least once: 6,398


### Training groups and chronological tuning slice

Training rows come from the full `labels_train` pool: each clicked positive is
paired with up to four exposed non-clicks redrawn from the same request each
epoch. The last 10% of training timestamps are held out for early stopping; the tuning slates come from
the complete `candidates_train.parquet`. Popularity is not attached to training
or tuning rows; it is used only at retrieval for ties and fallback.


In [12]:
run_manifest = json.loads((FEATURES / "run_manifest.json").read_text())
if run_manifest.get("training_candidate_pool") != "complete":
    raise ValueError(
        "Per-epoch sampling needs the complete training pool. "
        "Rebuild stage 4 with the default training_negative_sampling='epoch'."
    )
CFG = Settings(**run_manifest["config"])

TRAINING_ROWS = (
    pl.read_parquet(sorted((FEATURES / "labels_train").glob("*.parquet")))
    .select("impression_uid", "news_code", "clicked")
    .to_pandas()
)
if TRAINING_ROWS["clicked"].isna().any() or not TRAINING_ROWS[
    "clicked"
].isin([0, 1]).all():
    raise ValueError("Training rows must be uniquely binary-labelled")

train_profile_index = pl.read_parquet(
    sorted((FEATURES / f"request_profiles_{FIT_SPLIT}").glob("*.parquet")),
    columns=["impression_uid", "timestamp"],
)
profile_lookup = PROFILE_INDEX_TRAIN.join(
    train_profile_index, on="impression_uid", how="left"
)

positive_uids = np.unique(
    TRAINING_ROWS.loc[TRAINING_ROWS["clicked"].eq(1), "impression_uid"].to_numpy()
)
positive_profiles = profile_lookup.filter(
    pl.col("impression_uid").is_in(positive_uids)
)
cutoff_ns = chronological_cutoff(
    positive_profiles["timestamp"].to_numpy(), TUNE_QUANTILE
)
fit_profiles = chronological_mask(profile_lookup["timestamp"].to_numpy(), cutoff_ns)
fit_uids = set(
    profile_lookup.filter(pl.Series("fit", fit_profiles))["impression_uid"].to_list()
)
FIT_ROWS = TRAINING_ROWS[
    TRAINING_ROWS["impression_uid"].isin(fit_uids)
].reset_index(drop=True)
ALL_ROWS = TRAINING_ROWS
n_groups = int(TRAINING_ROWS["clicked"].sum())
EPOCH_SAMPLER = EpochNegativeSampler(ALL_ROWS, CFG)


def sample_epoch(epoch, *, fit_only):
    """Draw exposed negatives for one epoch and build combined-model arrays."""
    groups, _ = EPOCH_SAMPLER.draw(
        int(epoch), uids=fit_uids if fit_only else None
    )
    groups = (
        pl.from_pandas(groups)
        .join(profile_lookup, on="impression_uid", how="left")
        .with_row_index("group_row")
        .join(
            article_index.select(
                pl.col("news_code"),
                pl.col("article_row").alias("pos_row"),
            ),
            left_on="positive_news_code",
            right_on="news_code",
            how="left",
        )
    )
    if groups["profile_row"].null_count() or groups["pos_row"].null_count():
        raise ValueError("Sampled groups are missing training profiles or article rows")
    negative_long = (
        groups.select("group_row", "negative_news_codes")
        .explode("negative_news_codes")
        .rename({"negative_news_codes": "negative_news_code"})
        .drop_nulls("negative_news_code")
        .with_columns(pl.int_range(pl.len()).over("group_row").alias("neg_pos"))
        .join(
            article_index.select(
                pl.col("news_code").alias("negative_news_code"),
                pl.col("article_row").alias("neg_row"),
            ),
            on="negative_news_code",
            how="left",
        )
        .sort(["group_row", "neg_pos"])
    )
    if negative_long["neg_row"].null_count():
        raise ValueError("Sampled negatives are missing article rows")
    n_epoch_groups = groups.height
    NEG_MAX = int(groups["negative_news_codes"].list.len().max() or 0)
    if n_epoch_groups and NEG_MAX == 0:
        raise ValueError("Sampled training rows contain no exposed negatives")
    NEG_MATRIX = np.zeros((n_epoch_groups, NEG_MAX), dtype=np.int64)
    group_rows = negative_long["group_row"].to_numpy().astype(np.int64)
    NEG_MATRIX[
        group_rows, negative_long["neg_pos"].to_numpy().astype(np.int64)
    ] = negative_long["neg_row"].to_numpy().astype(np.int64)
    NEG_LEN = np.bincount(group_rows, minlength=n_epoch_groups).astype(np.int64)
    return (
        groups["profile_row"].to_numpy().astype(np.int32),
        groups["pos_row"].to_numpy().astype(np.int32),
        NEG_MATRIX,
        NEG_LEN,
    )


TUNE_NDCG_K = max_clicks_per_impression(TUNE_CANDIDATES)

tune = tuning_slates(
    TUNE_CANDIDATES,
    cutoff_ns,
    article_index,
    PROFILE_INDEX_TRAIN,
    max_impressions=None,
    seed=SEED + 1,
)
TUNE_PROFILE_ROWS = tune["profile_row"].to_numpy().astype(np.int32)
TUNE_ITEM_ROWS = tune["article_row"].to_numpy().astype(np.int32)
TUNE_CLICKED = tune["clicked"].to_numpy().astype(np.float32)
TUNE_UIDS = tune["impression_uid"].to_numpy()
print(f"Training rows: {len(TRAINING_ROWS):,}")
print(f"Training positives: {n_groups:,}")
print(f"Chronological fit rows: {len(FIT_ROWS):,} / {len(ALL_ROWS):,}")
print(f"Sampler negatives per positive: up to {CFG.negative_count}")
print("Tuning cutoff:", np.datetime64(cutoff_ns, "ns"))
print(f"Tuning impressions: {tune['impression_uid'].n_unique():,}")
print(f"Tuning pairs: {tune.height:,}")
print(f"Tuning NDCG k (max training clicks per impression): {TUNE_NDCG_K}")


Training rows: 3,389,342
Training positives: 141,558
Chronological fit rows: 3,046,331 / 3,389,342
Sampler negatives per positive: up to 4
Tuning cutoff: 2019-11-12T13:50:10.000000000
Tuning impressions: 9,507
Tuning pairs: 343,011
Tuning NDCG k (max training clicks per impression): 31


## 4. Setting up the model


### Two-tower network definition

`TwoTower` has two independent MLPs. The request tower consumes the pooled
166-dimensional history input; the item tower consumes the 166-dimensional
intrinsic content input. Both outputs are L2-normalised and combined by a
scaled dot product. There are no popularity, user-ID or article-ID embeddings.


In [13]:
class TwoTower(nn.Module):
    def __init__(
        self,
        user_input_dim=INPUT_DIM,
        item_input_dim=ITEM_INPUT_DIM,
        hidden_dim=256,
        embed_dim=128,
        dropout=0.1,
        scale=10.0,
        max_scale=100.0,
    ):
        super().__init__()
        self.user_tower = self._tower(
            user_input_dim, hidden_dim, embed_dim, dropout
        )
        self.item_tower = self._tower(
            item_input_dim, hidden_dim, embed_dim, dropout
        )
        self.logit_scale = nn.Parameter(
            torch.tensor(float(np.log(scale)), dtype=torch.float32)
        )
        self.max_log_scale = float(np.log(max_scale))

    @staticmethod
    def _tower(input_dim, hidden_dim, embed_dim, dropout):
        return nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, embed_dim),
        )

    def encode_users(self, values):
        return F.normalize(self.user_tower(values), dim=-1)

    def encode_items(self, values):
        return F.normalize(self.item_tower(values), dim=-1)

    def forward(self, user_values, item_values):
        scale = torch.exp(torch.clamp(self.logit_scale, max=self.max_log_scale))
        return scale * (
            self.encode_users(user_values) * self.encode_items(item_values)
        ).sum(dim=-1)


## 5. Training the model


### Two-tower training, chronological tuning and refit

The user tower consumes the 166-dimensional pooled history input. The item tower
consumes the 166-dimensional intrinsic content block only. The towers do not
share weights; their outputs are L2-normalised and combined by a scaled dot
product. Early stopping uses mean slate NDCG@k on the chronological tuning
slice, with k set to the maximum number of clicks per impression in the training
slates; the final model is then refit on all training groups for the selected
epoch count. Training uses BPR on the sampled negatives: each positive is
scored against the negatives redrawn for the same request each epoch.


In [14]:
def tune_ndcg(model):
    model.eval()
    scores = np.empty(len(TUNE_CLICKED), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, len(TUNE_CLICKED), 200_000):
            end = min(start + 200_000, len(TUNE_CLICKED))
            users = torch.as_tensor(
                np.array(
                    TRAIN_USER_FEATURES[TUNE_PROFILE_ROWS[start:end]],
                    dtype=np.float32,
                ),
                device=DEVICE,
            )
            items = torch.as_tensor(
                item_inputs(TUNE_ITEM_ROWS[start:end]),
                device=DEVICE,
            )
            scores[start:end] = model(users, items).cpu().numpy()
    _, ndcg = mean_slate_metrics(
        TUNE_UIDS, TUNE_CLICKED, scores, TUNE_NDCG_K
    )
    return ndcg


In [15]:
def train_epoch(
    model, optimizer, profile_rows, pos_item_rows, neg_matrix, neg_len, rng
):
    model.train()
    order = rng.permutation(len(profile_rows))
    total = 0.0
    for start in range(0, len(order), BATCH_SIZE):
        batch = order[start : start + BATCH_SIZE]
        users = torch.as_tensor(
            np.array(
                TRAIN_USER_FEATURES[profile_rows[batch]], dtype=np.float32
            ),
            device=DEVICE,
        )
        positives = torch.as_tensor(
            item_inputs(pos_item_rows[batch]), device=DEVICE
        )
        neg_rows_np = neg_matrix[batch]
        neg_lens_np = neg_len[batch]
        negatives = torch.as_tensor(
            neg_rows_np, dtype=torch.long, device=DEVICE
        )
        neg_lens = torch.as_tensor(
            neg_lens_np, dtype=torch.long, device=DEVICE
        )
        neg_mask = (
            torch.arange(negatives.shape[1], device=DEVICE)[None, :]
            < neg_lens[:, None]
        )
        users_expanded = users[:, None, :].expand(
            -1, negatives.shape[1], -1
        ).reshape(-1, users.shape[1])
        negative_inputs = torch.as_tensor(
            item_inputs(neg_rows_np.reshape(-1)), device=DEVICE
        )
        optimizer.zero_grad(set_to_none=True)
        positive_scores = model(users, positives)
        negative_scores = model(
            users_expanded, negative_inputs
        ).reshape(len(batch), negatives.shape[1])
        pair_loss = F.logsigmoid(
            positive_scores[:, None] - negative_scores
        )
        loss = -(pair_loss * neg_mask).sum() / neg_mask.sum()
        loss.backward()
        optimizer.step()
        total += float(loss.item()) * len(batch)
    return total / len(order)


In [16]:
def build_model():
    return TwoTower(
        user_input_dim=INPUT_DIM,
        item_input_dim=ITEM_INPUT_DIM,
        hidden_dim=HIDDEN_DIM,
        embed_dim=EMBED_DIM,
        dropout=DROPOUT,
    ).to(DEVICE)


In [17]:
model = build_model()

optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)

rng = np.random.default_rng(SEED + 1)

training_log = []

stopper = EarlyStopper(patience=PATIENCE)

for epoch in range(1, NUM_EPOCHS + 1):
    started = time.time()
    profile_rows, pos_rows, neg_matrix, neg_len = sample_epoch(
        epoch - 1, fit_only=True
    )
    loss = train_epoch(
        model,
        optimizer,
        profile_rows,
        pos_rows,
        neg_matrix,
        neg_len,
        rng,
    )
    ndcg = tune_ndcg(model)
    elapsed = time.time() - started
    training_log.append(
        {
            "phase": "tune",
            "epoch": epoch,
            "train_loss": loss,
            "tune_ndcg": ndcg,
            "ndcg_k": TUNE_NDCG_K,
            "seconds": elapsed,
        }
    )
    print(
        f"epoch {epoch:02d} loss={loss:.6f} "
        f"tune_ndcg@{TUNE_NDCG_K}={ndcg:.5f} ({elapsed:.1f}s)"
    )
    if stopper.step(ndcg, model, epoch):
        print("Early stopping")
        break

stopper.restore(model)

print(
    f"Best epoch: {stopper.best_epoch} "
    f"(tune_ndcg@{TUNE_NDCG_K}={stopper.best_score:.5f})"
)

final_model = build_model()

optimizer = torch.optim.AdamW(
    final_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)

rng = np.random.default_rng(SEED + 2)

for epoch in range(1, stopper.best_epoch + 1):
    started = time.time()
    profile_rows, pos_rows, neg_matrix, neg_len = sample_epoch(
        epoch - 1, fit_only=False
    )
    loss = train_epoch(
        final_model,
        optimizer,
        profile_rows,
        pos_rows,
        neg_matrix,
        neg_len,
        rng,
    )
    elapsed = time.time() - started
    training_log.append(
        {
            "phase": "refit",
            "epoch": epoch,
            "train_loss": loss,
            "tune_ndcg": float("nan"),
            "ndcg_k": TUNE_NDCG_K,
            "seconds": elapsed,
        }
    )
    print(f"refit epoch {epoch:02d} loss={loss:.6f} ({elapsed:.1f}s)")

model = final_model

torch.save(
    {
        "state_dict": model.state_dict(),
        "config": {
            "user_input_dim": INPUT_DIM,
            "item_input_dim": ITEM_INPUT_DIM,
            "hidden_dim": HIDDEN_DIM,
            "embed_dim": EMBED_DIM,
            "dropout": DROPOUT,
            "item_features": "text64 + entity100 + availability flags",
            "objective": "bpr",
            "max_epochs": NUM_EPOCHS,
            "best_epoch": stopper.best_epoch,
            "epochs": stopper.best_epoch,
            "tune_ndcg": stopper.best_score,
            "ndcg_k": TUNE_NDCG_K,
            "batch_size": BATCH_SIZE,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "seed": SEED,
            "training_data": "labels_train full training pool",
            "train_groups": int(n_groups),
            "negative_sampling": "per-epoch exposed negatives",
            "negative_count": int(CFG.negative_count),
            "sampler_epoch_base": 0,
            "fit_rows": int(len(FIT_ROWS)),
            "refit_rows": int(len(ALL_ROWS)),
        },
    },
    ARTIFACT_DIR / "model.pt",
)

pl.DataFrame(training_log).write_csv(ARTIFACT_DIR / "training_log.csv")

print("Saved model:", ARTIFACT_DIR / "model.pt")


C:\Users\Yong Rui\AppData\Local\Temp\ipykernel_8200\1487132553.py:71: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  .explode("negative_news_codes")


epoch 01 loss=0.677759 tune_ndcg@31=0.42394 (9.0s)


C:\Users\Yong Rui\AppData\Local\Temp\ipykernel_8200\1487132553.py:71: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  .explode("negative_news_codes")


epoch 02 loss=0.636764 tune_ndcg@31=0.43178 (3.9s)


C:\Users\Yong Rui\AppData\Local\Temp\ipykernel_8200\1487132553.py:71: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  .explode("negative_news_codes")


epoch 03 loss=0.620609 tune_ndcg@31=0.44571 (3.9s)


... (88 outputs left out)

C:\Users\Yong Rui\AppData\Local\Temp\ipykernel_8200\1487132553.py:71: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  .explode("negative_news_codes")


refit epoch 20 loss=0.530216 (3.3s)


C:\Users\Yong Rui\AppData\Local\Temp\ipykernel_8200\1487132553.py:71: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  .explode("negative_news_codes")


refit epoch 21 loss=0.528614 (3.2s)
Saved model: c:\Users\Yong Rui\BT4222-Group-6-\5_Models\5_01_Candidate_Generation\artifacts\combined_content_two_tower_rolling_popularity\model.pt


## 6. Combined candidate generation

For each request, the popularity half selects the first 750 non-history articles from the timestamp's causal popularity order. The content half scores the full eligible catalogue with the content two-tower, excludes history and the popularity half, and selects 750 articles by content score. Score ties and empty-profile requests use the content model's rolling prior-hour click order.


### Candidate helpers

These helpers keep the candidate format, eligibility policy, history exclusion
and content tie-breaking consistent with `CG_content_two_tower_mlp.ipynb`.


In [18]:
def popularity_select(order, n_eligible, excluded=None, k=CANDIDATE_CAP):
    excluded_set = set() if excluded is None else set(map(int, excluded))
    selected = []
    for position in order:
        position = int(position)
        if position >= n_eligible or position in excluded_set:
            continue
        selected.append(position)
        if len(selected) == k:
            break
    return np.asarray(selected, dtype=np.int64)


In [19]:
def popularity_fill(available, popularity, news_code, count):
    pool = np.flatnonzero(available)
    order = np.lexsort((news_code[pool], -popularity[pool]))
    return pool[order[:count]]


In [20]:
def select_top_k(
    scores, popularity, news_code, k, excluded=None, fallback_order=None
):
    scores = np.asarray(scores, dtype=np.float32)
    popularity = np.asarray(popularity)
    news_code = np.asarray(news_code)
    n = scores.size
    if fallback_order is not None:
        return popularity_select(fallback_order, n, excluded=excluded, k=k)

    available = np.ones(n, dtype=bool)
    if excluded is not None and len(excluded):
        positions = np.asarray(excluded, dtype=np.int64)
        available[positions] = False
        scores = scores.copy()
        scores[positions] = -np.inf

    finite = np.isfinite(scores)
    n_finite = int(np.count_nonzero(finite))
    if n_finite == 0:
        return popularity_fill(available, popularity, news_code, k)

    take = min(k, n_finite)
    if take < n_finite:
        frontier = np.argpartition(-scores, take - 1)[:take]
        threshold = scores[frontier].min()
    else:
        threshold = scores[finite].min()

    above = np.flatnonzero(scores > threshold)
    need = take - above.size
    ties = np.flatnonzero(scores == threshold)
    if need > 0:
        if ties.size > need:
            tie_order = np.lexsort((news_code[ties], -popularity[ties]))
            ties = ties[tie_order[:need]]
        chosen_positions = np.r_[above, ties]
    else:
        chosen_positions = above

    order = np.lexsort(
        (
            news_code[chosen_positions],
            -popularity[chosen_positions],
            -scores[chosen_positions],
        )
    )
    selected = chosen_positions[order].astype(np.int64)

    if selected.size < k:
        remaining = available.copy()
        remaining[selected] = False
        fill = popularity_fill(remaining, popularity, news_code, k - selected.size)
        selected = np.r_[selected, fill]

    return selected


In [21]:
def encode_user_batches(model, features, batch=100_000):
    out = np.empty((len(features), EMBED_DIM), dtype=np.float32)
    model.eval()
    with torch.no_grad():
        for start in range(0, len(features), batch):
            end = min(start + batch, len(features))
            values = torch.from_numpy(
                np.array(features[start:end], dtype=np.float32, copy=True)
            ).to(DEVICE)
            out[start:end] = model.encode_users(values).cpu().numpy()
    return out


In [22]:
def encode_item_batches(model, features, batch=100_000):
    out = np.empty((len(features), EMBED_DIM), dtype=np.float32)
    model.eval()
    with torch.no_grad():
        for start in range(0, len(features), batch):
            end = min(start + batch, len(features))
            values = torch.from_numpy(
                np.array(features[start:end], dtype=np.float32, copy=True)
            ).to(DEVICE)
            out[start:end] = model.encode_items(values).cpu().numpy()
    return out


In [23]:
target_profiles = load_profiles(TARGET_SPLIT)
TARGET_USER_FEATURES, _ = load_profile_cache(TARGET_SPLIT)

model = final_model
model.eval()
user_emb = encode_user_batches(model, TARGET_USER_FEATURES)
item_emb = encode_item_batches(model, CONTENT_INPUTS_SORTED)

has_profile = (
    (TARGET_USER_FEATURES[:, -2] > 0) | (TARGET_USER_FEATURES[:, -1] > 0)
)

impression_uids = target_profiles["impression_uid"].to_numpy()
timestamps = target_profiles["timestamp"].to_numpy().astype("datetime64[ns]")
ts_ns = timestamps.astype("datetime64[ns]").astype(np.int64)
TARGET_HOUR_INDEX = hour_index_of(ts_ns)
history_rows = target_profiles["full_history_rows"].to_list()

starts = np.flatnonzero(np.r_[True, timestamps[1:] != timestamps[:-1]])
ends = np.r_[starts[1:], len(timestamps)]
batches = list(zip(starts, ends))
print(f"Scoring requests: {len(timestamps):,}")


def select_popularity_prefix(
    ranked_rows,
    ranked_pos,
    n_eligible,
    history_mask,
    k,
    initial=POPULARITY_SCAN,
):
    pool = min(len(ranked_rows), max(initial, k + 512))
    while True:
        rows = ranked_rows[:pool]
        pos = ranked_pos[:pool]
        ok = (pos < n_eligible) & ~history_mask[rows]
        selected_rows = rows[ok]
        selected_pos = pos[ok]
        if selected_rows.size >= k or pool >= len(ranked_rows):
            return selected_rows[:k], selected_pos[:k]
        pool = min(len(ranked_rows), pool * 2)


cutoff_values = np.arange(1, CANDIDATE_CAP + 1, dtype=np.int16)
writer = CandidateWriter(RECOMMENDATIONS_PATH)
current_hour = None
ranked_rows = None
ranked_pos = None
popularity_sorted = None
fallback_order = None
started = time.time()
try:
    for batch_no, (start, end) in enumerate(batches, 1):
        timestamp = timestamps[start]
        batch_hour = int(TARGET_HOUR_INDEX[start])
        if batch_hour != current_hour:
            ranked_rows, ranked_pos = popularity_snapshot(batch_hour)
            popularity_sorted = hourly_popularity(
                batch_hour, ARTICLE_ROWS_SORTED
            )
            fallback_order = np.lexsort(
                (NEWS_SORTED, -popularity_sorted)
            )
            current_hour = batch_hour

        n_eligible = int(
            np.searchsorted(AVAILABLE_SORTED, timestamp, side="right")
        )

        scores = user_emb[start:end] @ item_emb[:n_eligible].T
        popularity_batch = popularity_sorted[:n_eligible]
        news_batch = NEWS_SORTED[:n_eligible]

        selected_batch = np.empty(
            (end - start, CANDIDATE_CAP), dtype=np.int32
        )
        for local, profile_row in enumerate(range(start, end)):
            history = history_rows[profile_row]
            history_mask = np.zeros(n_items, dtype=bool)
            if history:
                history_mask[np.asarray(history, dtype=np.int64)] = True

            pop_rows, pop_pos = select_popularity_prefix(
                ranked_rows,
                ranked_pos,
                n_eligible,
                history_mask,
                POPULARITY_BUDGET,
            )
            pop_news = row_to_code[pop_rows]
            history_pos = (
                POS_IN_SORTED[np.asarray(history, dtype=np.int64)]
                if history
                else np.empty(0, dtype=np.int64)
            )
            history_pos = history_pos[
                (history_pos >= 0) & (history_pos < n_eligible)
            ]
            excluded = np.union1d(history_pos, pop_pos)

            fallback = None if has_profile[profile_row] else fallback_order
            content_local = select_top_k(
                scores[local],
                popularity_batch,
                news_batch,
                CONTENT_BUDGET,
                excluded=excluded,
                fallback_order=fallback,
            )
            content_news = news_batch[content_local]

            selected_batch[local] = np.r_[pop_news, content_news]

        uids = impression_uids[start:end]
        writer.add(
            np.repeat(uids, CANDIDATE_CAP),
            np.tile(cutoff_values, end - start),
            selected_batch.reshape(-1),
        )
        if batch_no % 1000 == 0 or batch_no == len(batches):
            print(
                f"Scored {batch_no:,}/{len(batches):,} timestamp batches; "
                f"{writer.rows:,} rows",
                flush=True,
            )
finally:
    writer.close()

elapsed = time.time() - started
print(f"Wrote {writer.rows:,} combined rows in {elapsed:.1f}s")
print("Combined candidates:", RECOMMENDATIONS_PATH)


Built profile cache for validation: 31,624 requests
Scoring requests: 31,624
Scored 1,000/24,580 timestamp batches; 523,500 rows
Scored 2,000/24,580 timestamp batches; 1,099,000 rows
Scored 3,000/24,580 timestamp batches; 1,718,000 rows
Scored 4,000/24,580 timestamp batches; 2,367,000 rows
Scored 5,000/24,580 timestamp batches; 3,048,000 rows
Scored 6,000/24,580 timestamp batches; 3,734,000 rows
Scored 7,000/24,580 timestamp batches; 4,404,000 rows
Scored 8,000/24,580 timestamp batches; 5,095,500 rows
Scored 9,000/24,580 timestamp batches; 5,771,000 rows
Scored 10,000/24,580 timestamp batches; 6,465,500 rows
Scored 11,000/24,580 timestamp batches; 7,188,000 rows
Scored 12,000/24,580 timestamp batches; 7,901,000 rows
Scored 13,000/24,580 timestamp batches; 8,620,500 rows
Scored 14,000/24,580 timestamp batches; 9,331,000 rows
Scored 15,000/24,580 timestamp batches; 10,035,500 rows
Scored 16,000/24,580 timestamp batches; 10,730,000 rows
Scored 17,000/24,580 timestamp batches; 11,417,000 r

## 7. Evaluation

The combined candidate set is evaluated with the shared `CandidateEvaluator`.


In [24]:
from cg_evaluation import CandidateEvaluator

evaluator = CandidateEvaluator(root=ROOT, split=TARGET_SPLIT)
evaluation = evaluator.evaluate_to_csv(
    RECOMMENDATIONS_PATH,
    model_name=MODEL_NAME,
    m_values=M_VALUES,
    output_dir=RESULTS,
)
print(evaluation.sort("m"))
print("Results:", RESULT_PATH)


shape: (8, 7)
┌───────────────────┬────────────┬─────┬───────────────────┬───────────┬─────────────┬─────────────┐
│ model             ┆ split      ┆ m   ┆ metric            ┆ value     ┆ numerator   ┆ denominator │
│ ---               ┆ ---        ┆ --- ┆ ---               ┆ ---       ┆ ---         ┆ ---         │
│ str               ┆ str        ┆ i64 ┆ str               ┆ f64       ┆ f64         ┆ f64         │
╞═══════════════════╪════════════╪═════╪═══════════════════╪═══════════╪═════════════╪═════════════╡
│ content_two_tower ┆ validation ┆ 500 ┆ catalogue_coverag ┆ 0.520047  ┆ 24722.0     ┆ 47538.0     │
│ _rolling_popu…    ┆            ┆     ┆ e                 ┆           ┆             ┆             │
│ content_two_tower ┆ validation ┆ 500 ┆ cold_article_reca ┆ 0.035088  ┆ 12.0        ┆ 342.0       │
│ _rolling_popu…    ┆            ┆     ┆ ll                ┆           ┆             ┆             │
│ content_two_tower ┆ validation ┆ 500 ┆ hit_rate          ┆ 0.864185  ┆ 2732

## Notes and limitations

- The two halves are disjoint by construction: the content half excludes the 750
  popularity candidates and both halves exclude `full_history_rows`.
- The popularity half is a history-filtered version of the rolling
  `prior_hour_clicks` order. The existing baseline Top-750 artifact does not
  exclude history, so minor differences are expected.
- The content half scores the full eligible catalogue. This is deliberately
  different from the popularity-pool rerankers: it can surface niche articles
  that the popularity head never reaches.
- Recall ignores candidate order. Cutoffs `1-750` identify the popularity half;
  `751-1500` identify the content half. The downstream reranker receives the
  union.
- The content model is trained with BPR on grouped exposed negatives (up to four
  per positive), so its scores are ranking scores, not calibrated click
  probabilities.
- `available_from` is the project-wide observable-availability proxy, not a
  verified publication timestamp.
